# Apple Global Sales — ETL Pipeline

End-to-end data pipeline: raw CSV → cleaning → star schema → SQL Server data warehouse.

**Stack:** Python · pandas · SQLAlchemy · SQL Server

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

## 2. Load raw data

In [ ]:
df = pd.read_csv("apple_global_sales_dataset.csv")
print("Shape:", df.shape)
df.head()

## 3. Explore nulls

In [ ]:
nulls = df.isnull().sum()
nulls[nulls > 0]

## 4. Handle nulls

- `storage` and `previous_device_os` are text columns — fill with `'Unknown'`
- `customer_rating` is numeric — fill with median

In [ ]:
df['storage'].fillna('Unknown', inplace=True)
df['previous_device_os'].fillna('Unknown', inplace=True)
df['customer_rating'].fillna(df['customer_rating'].median(), inplace=True)

print('Nulls remaining:', df.isnull().sum().sum())

## 5. Fix data types

In [ ]:
df['sale_date'] = pd.to_datetime(df['sale_date'])
df['unit_price_usd'] = df['unit_price_usd'].astype(float)
df['revenue_usd'] = df['revenue_usd'].astype(float)
df['units_sold'] = df['units_sold'].astype(int)
df['discount_pct'] = df['discount_pct'].astype(int)

## 6. Standardize text columns

In [ ]:
text_cols = [
    'country', 'region', 'city', 'product_name', 'category',
    'sales_channel', 'payment_method', 'customer_segment',
    'customer_age_group', 'return_status', 'color',
    'storage', 'previous_device_os'
]

for col in text_cols:
    df[col] = df[col].str.strip().str.title()

## 7. Feature engineering

- `revenue_after_discount` — actual revenue after applying discount
- `is_returned` — binary flag for returned orders
- `is_high_value` — binary flag for transactions over $1,000
- `day_of_week` and `week_number` — for time intelligence in Power BI

In [ ]:
df['revenue_after_discount'] = df['units_sold'] * df['discounted_price_usd']
df['is_returned'] = df['return_status'].apply(lambda x: 1 if x == 'Returned' else 0)
df['is_high_value'] = df['revenue_usd'].apply(lambda x: 1 if x >= 1000 else 0)
df['day_of_week'] = df['sale_date'].dt.day_name()
df['week_number'] = df['sale_date'].dt.isocalendar().week.astype(int)

df.head()

## 8. Build star schema

### Dimension tables

| Table | Key | Description |
|-------|-----|-------------|
| dim_product | product_id | Product name, category, storage, color |
| dim_location | location_id | Country, region, city, currency, FX rate |
| dim_customer | customer_id | Segment, age group, previous OS |
| dim_date | date_id | Date, year, quarter, month, day, week |

In [ ]:
# dim_product — unique product variants (no unit_price_usd: price varies per transaction)
dim_product = df[['product_name', 'category', 'storage', 'color']]\
    .drop_duplicates().reset_index(drop=True)
dim_product['product_id'] = dim_product.index + 1

# dim_location
dim_location = df[['country', 'region', 'city', 'currency', 'fx_rate_to_usd']]\
    .drop_duplicates().reset_index(drop=True)
dim_location['location_id'] = dim_location.index + 1

# dim_customer
dim_customer = df[['customer_segment', 'customer_age_group', 'previous_device_os']]\
    .drop_duplicates().reset_index(drop=True)
dim_customer['customer_id'] = dim_customer.index + 1

# dim_date
dim_date = df[['sale_date', 'year', 'quarter', 'month', 'day_of_week', 'week_number']]\
    .drop_duplicates().reset_index(drop=True)
dim_date['date_id'] = dim_date.index + 1

print('dim_product:', dim_product.shape)
print('dim_location:', dim_location.shape)
print('dim_customer:', dim_customer.shape)
print('dim_date:', dim_date.shape)

### Fact table

The central fact table joins all dimension keys and keeps all transactional measures.

In [ ]:
fact = df.merge(dim_product, on=['product_name', 'category', 'storage', 'color'])\
         .merge(dim_location, on=['country', 'region', 'city', 'currency', 'fx_rate_to_usd'])\
         .merge(dim_customer, on=['customer_segment', 'customer_age_group', 'previous_device_os'])\
         .merge(dim_date, on=['sale_date', 'year', 'quarter', 'month', 'day_of_week', 'week_number'])

fact_sales = fact[[
    'sale_id', 'product_id', 'location_id', 'customer_id', 'date_id',
    'units_sold', 'unit_price_usd', 'discount_pct', 'discounted_price_usd',
    'revenue_usd', 'revenue_local_currency', 'revenue_after_discount',
    'payment_method', 'sales_channel', 'customer_rating',
    'return_status', 'is_returned', 'is_high_value'
]]

print('fact_sales:', fact_sales.shape)

## 9. Load into SQL Server



In [ ]:
engine = create_engine(
    'mssql+pyodbc://localhost\\SQLEXPRESS/applle_datawarehouse'
    '?driver=ODBC+Driver+17+for+SQL+Server'
    '&trusted_connection=yes'
)

dim_product.to_sql('dim_product', engine, if_exists='replace', index=False)
dim_location.to_sql('dim_location', engine, if_exists='replace', index=False)
dim_customer.to_sql('dim_customer', engine, if_exists='replace', index=False)
dim_date.to_sql('dim_date', engine, if_exists='replace', index=False)
fact_sales.to_sql('fact_sales', engine, if_exists='replace', index=False)

print('All tables loaded successfully!')

## 10. Verify row counts

In [ ]:
verification = pd.read_sql("""
    SELECT 'fact_sales'   AS table_name, COUNT(*) AS rows FROM fact_sales
    UNION ALL
    SELECT 'dim_product',  COUNT(*) FROM dim_product
    UNION ALL
    SELECT 'dim_location', COUNT(*) FROM dim_location
    UNION ALL
    SELECT 'dim_customer', COUNT(*) FROM dim_customer
    UNION ALL
    SELECT 'dim_date',     COUNT(*) FROM dim_date
""", engine)

print(verification.to_string(index=False))